In [1]:
# 1. Instalar as bibliotecas necessárias
!pip install -q torch torchvision open_clip_torch gradio pillow matplotlib

import torch
import open_clip
from PIL import Image
import matplotlib.pyplot as plt
import gradio as gr
import numpy as np

# 2. Configurar o dispositivo (usa T4 GPU se disponível no Colab)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"➔ Dispositivo em uso: {device}")

# 3. Carregar o modelo BiomedCLIP
print("➔ A carregar o BiomedCLIP... (Aguarde alguns segundos)")
model, _, preprocess = open_clip.create_model_and_transforms(
    'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224',
    device=device
)
tokenizer = open_clip.get_tokenizer('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')
model.eval()
print("➔ Modelo carregado com sucesso!")

# Classes de diagnóstico (baseadas no projeto CELLo)
classes = [
    "Saudável / Sem Anomalia (NA)",
    "Estágio Tumoral Baixo (Ta)",
    "Estágio Tumoral T1",
    "Estágio Tumoral T2",
    "Estágio Tumoral Avançado (T3/T4)"
]

def diagnosticar_imagem(imagem):
    """Função acionada quando o utilizador faz upload de uma imagem na interface."""
    if imagem is None:
        return None

    # Converter imagem para formato PIL compatível
    if isinstance(imagem, np.ndarray):
        image = Image.fromarray(imagem).convert("RGB")
    else:
        image = imagem.convert("RGB")

    # Pré-processamento e inferência
    image_tensor = preprocess(image).unsqueeze(0).to(device)
    text_tokens = tokenizer(classes).to(device)

    with torch.no_grad():
        image_features = model.encode_image(image_tensor)
        text_features = model.encode_text(text_tokens)

        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)

        text_probs = (100.0 * image_features @ text_features.T).softmax(dim=-1).cpu().numpy()[0]

    # Criar o gráfico de barras com Matplotlib
    fig, ax = plt.subplots(figsize=(6, 4))
    pares = sorted(zip(classes, text_probs), key=lambda x: x[1])
    sorted_classes, sorted_probs = zip(*pares)

    y_pos = range(len(classes))
    barras = ax.barh(y_pos, [p * 100 for p in sorted_probs], color="#2980b9")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(sorted_classes, fontsize=9)
    ax.set_xlabel("Probabilidade de Diagnóstico (%)", fontsize=10, fontweight="bold")
    ax.set_title("Confiança do BiomedCLIP (Zero-Shot)", fontsize=11, fontweight="bold")
    ax.set_xlim(0, 100)

    for barra in barras:
        width = barra.get_width()
        ax.text(width + 1, barra.get_y() + barra.get_height()/2, f"{width:.1f}%", va="center", fontsize=9, fontweight="bold")

    plt.tight_layout()
    return fig

# 4. Criar a Interface Gráfica Web com Gradio
demo = gr.Interface(
    fn=diagnosticar_imagem,
    inputs=gr.Image(type="pil", label="Carregar Imagem Médica (Ex: Citologia)"),
    outputs=gr.Plot(label="Gráfico de Probabilidades do Modelo"),
    title="CELLo — Sistema de Diagnóstico com BiomedCLIP",
    description="Carregue ou arraste uma imagem citológica para a caixa abaixo para obter o diagnóstico automático em tempo real.",
    theme="default"
)

# Executar a aplicação interativa no Colab
demo.launch(inline=True, share=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.3 MB/s eta 0:00:00
➔ Dispositivo em uso: cpu
➔ A carregar o BiomedCLIP... (Aguarde alguns segundos)


open_clip_config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

open_clip_pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  784MB            

open_clip_pytorch_model.bin: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/225k [00:00<?, ?B/s]

➔ Modelo carregado com sucesso!


/usr/local/lib/python3.13/dist-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7fa43eb6008d74a548.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
